In [0]:
from pathlib import Path

repo_root = Path.cwd().parent     
log_dir = repo_root / "lammps" / "Si" / "logs"

print(log_dir)

In [0]:
from pathlib import Path
import re

from pyspark.sql import Row
from pyspark.sql.functions import col, regexp_extract

rows = []

for log_file in log_dir.glob("*.log"):

    # Extract temperature from filename
    temperature = int(re.search(r"(\d+)", log_file.stem).group(1))

    # Read log
    df = spark.read.text(str(log_file))

    # Extract lattice parameter
    lp = (
        df.filter(col("value").rlike(r"^lattice parameter equals"))
          .select(
              regexp_extract("value", r"(\d+\.\d+)", 1)
              .cast("double")
              .alias("lattice_parameter")
          )
          .first()["lattice_parameter"]
    )

    rows.append(
        Row(
            temperature=temperature,
            lattice_parameter=lp
        )
    )

# Create Spark DataFrame
silver_df = spark.createDataFrame(rows)

silver_df.orderBy("temperature").show()

In [0]:
%sql

-- Creating schema for silicon related simulations
CREATE SCHEMA silicon;

In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silicon.lattice_parameter")

In [0]:
%sql

select * from silicon.lattice_parameter order by temperature;